# Expressions - JavaScript

All 3 JavaScript examples from [docs/expression.md](https://platob.github.io/yggdryl/expression/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

## The four stages

In [ ]:
const assert = require('node:assert/strict')
const { Expression, Field, Value } = require('yggdryl')

const schema = new Field('trades', 'struct<ccy:utf8,price:decimal(9,2),size:bigint>', false)
const filter = new Expression("ccy = 'EUR' and price > 100")

assert.equal(filter.toString(), "ccy = 'EUR' and price > 100")
assert.deepEqual(filter.columns, ['ccy', 'price'])

const bound = filter.bind(schema)
assert.equal(
  bound.expression.toString(),
  "ccy = 'EUR' and price > decimal128(9,2) '100.00'",
)

// The price is an exact decimal, because the column is exact and so is
// the comparison: a JavaScript number here would be a different one.
const price = Value.d128(15000n, 2)
assert.equal(bound.matches(Value.fromJs(['EUR', price, 5])), true)
assert.equal(bound.matches(Value.fromJs(['USD', price, 5])), false)

## `&holder.*`: asking about the file

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
for (const year of ['2024', '2025']) {
  fs.mkdirSync(path.join(root, `year=${year}`))
  fs.writeFileSync(path.join(root, `year=${year}`, 'part-0.parquet'), '')
}

const lake = new IOBase(root)
const matched = [...lake.childrenMatching("&holder.partition['year'] = '2024'")]
assert.ok(matched.length > 0)
for (const entry of matched) {
  assert.match(String(entry.url), /year=2024/)
}

fs.rmSync(root, { recursive: true, force: true })

## Bind a whole statement once

In [ ]:
const assert = require('node:assert/strict')
const { Field, Statement } = require('yggdryl')

const field = new Field('rows', 'struct<ccy:utf8,size:bigint>', false)
const bound = new Statement(
  'select ccy, size as quantity where size >= :floor limit 10',
).bind(field, { floor: 2 })
assert.equal(bound.output.name, 'rows')